In [15]:
# !pip install -q langchain langchain-community langchain-openai faiss-cpu pymupdf
# !pip install -q fastapi uvicorn requests

In [16]:
from dotenv import load_dotenv

load_dotenv()

True

# Load

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

PDF_PATH = "../data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf"

loadar = PyMuPDFLoader(PDF_PATH)
pages = loadar.load()

print(len(pages))                  
print(pages[1].page_content[:200]) 

30
08
생성형 AI윤리 가이드북
PART
생성형 AI란?
01
생성형 AI가 무엇일까요?
생성형 AI(Generative AI)란, 대규모 데이터와 패턴을 학습하고 기존의 데이터를 활용하여 이용자의 요구에 
따라 텍스트, 이미지, 비디오, 음악, 코딩 등 새로운 결과를 만들어 내는 인공지능 기술입니다. 생성형 AI는 
누구나 쉽게 활용할 수 있도록 설계되었고


# Text Split

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

split_pages = text_splitter.split_documents(pages)

print(len(split_pages))

84


# Embedding & Store

In [19]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(split_pages, embeddings)

# Retriever

In [20]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 4})

found = retriever.invoke("세계보건기구(WHO)가 의료 분야에서 생성형 AI 활용 시 주의할 점으로 지적한 내용은 무엇인가요?")
for i, d in enumerate(found, 1):
    print(i, d.metadata.get("page"), d.page_content[:60].replace("\n", " "))

1 27 못한 문제를 막아야 합니다.  관련사례 ▶ 인공지능 의사, 자칫 ‘돌팔이’로…WHO “의료분야 활용 땐 검
2 26 예측하지 못한 손해를 유발할 수 있어요. 또한 생성형 AI는 아직 확률이라는 것을 반영해 세심하게 표현하는 
3 26 책임이 있음을 분명히 한 것임. 4  생성형 AI가 전문적인 정보까지 알려주는 것 같더라고요. 심리, 의료
4 25 기술이기 때문에 잘 모르는 내용이라도 이야기를 지어내 그럴듯하게 설명한다”라며 “이런 점에서  ‘확률적 앵무


# Prompt & LLM

In [21]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template(
    """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved [context] to answer the [question]. 
If you don't know the [answer] in [context], just say that you don't know the [answer]. 
Answer in Korean.

[context]
{context}

[question]
{question}

[answer]"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Chain

In [22]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(pages):
    return "\n\n".join(d.page_content for d in pages)

rag_chain = (
    {"context": retriever | format_docs,
    "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)